In [ ]:
from pyspark.sql.functions import coalesce, col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_products_table_name = dbutils.widgets.get("raw_olist_products_table")
raw_category_translation_table_name = dbutils.widgets.get("raw_category_translation_table")

silver_schema = dbutils.widgets.get("silver_schema")
products_table_name = dbutils.widgets.get("products_table")

In [ ]:
raw_olist_products_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_products_table_name}")
raw_category_translation_df = spark.table(f"{catalog}.{bronze_schema}.{raw_category_translation_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{products_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{products_table_name} (
            productId STRING,
            productCategoryName STRING,
            productNameLength INT,
            productDescriptionLength INT,
            productPhotosQty INT,
            productWeightG INT,
            productLengthCm INT,
            productHeightCm INT,
            productWidthCm INT,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
stage_category_translation_df = raw_category_translation_df.dropDuplicates(["product_category_name"])

In [ ]:
products_silver_df = with_processed_timestamp(
    raw_olist_products_df.alias("products")
    .join(stage_category_translation_df.alias("category"), on="product_category_name", how="left")
    .where(is_valid_uuid("products.product_id"))
    .select(
        col("products.product_id").cast("string").alias("productId"),
        coalesce(col("category.product_category_name_english"), col("products.product_category_name"))
        .cast("string")
        .alias("productCategoryName"),
        col("products.product_name_lenght").cast("int").alias("productNameLength"),
        col("products.product_description_lenght").cast("int").alias("productDescriptionLength"),
        col("products.product_photos_qty").cast("int").alias("productPhotosQty"),
        col("products.product_weight_g").cast("int").alias("productWeightG"),
        col("products.product_length_cm").cast("int").alias("productLengthCm"),
        col("products.product_height_cm").cast("int").alias("productHeightCm"),
        col("products.product_width_cm").cast("int").alias("productWidthCm"),
    )
    .dropDuplicates(["productId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{products_table_name}",
    source_view="products_silver_view",
    keys=["productId"],
    source_df=products_silver_df,
)